# Testes de persistência — Biblioteca
Java + ORMLite + SQLite. Execute `mvn package dependency:copy-dependencies -DincludeScope=runtime` na raiz e abra este notebook a partir de `notebooks/`, com o kernel Java (IJava). Reinicie o kernel antes de executar tudo novamente. Cada execução cria um banco temporário novo; nenhum banco existente é apagado.

In [1]:
%classpath add jar ../target/biblioteca-1.0.0.jar
%classpath add jar ../target/dependency/*.jar

In [2]:
import br.edu.biblioteca.*;
import java.time.LocalDate;
import java.nio.file.Files;
import java.sql.SQLException;
void verificar(boolean condicao, String descricao) {
    if (!condicao) throw new AssertionError(descricao);
    System.out.println("OK: " + descricao);
}
String arquivo = Files.createTempFile("biblioteca-", ".db").toString();
Database db = new Database(arquivo);
BibliotecaService servico = new BibliotecaService(db);
System.out.println("Banco de teste: " + arquivo);

Banco de teste: /tmp/biblioteca-8891469874607388262.db


## Cadastro e relação 1:1
O leitor pode ter zero ou uma carteirinha; cada carteirinha pertence a um único leitor.

In [3]:
Leitor leitor = new Leitor();
leitor.setNome("Bárbara"); leitor.setEmail("barbara@example.com");
db.dao(Leitor.class).create(leitor);
Carteirinha cartao = new Carteirinha();
cartao.setLeitor(leitor); cartao.setNumero("C001"); cartao.setEmitidaEm("2026-09-09");
db.dao(Carteirinha.class).create(cartao);
verificar(db.dao(Carteirinha.class).queryForId(cartao.getId()).getLeitor().getNome().equals("Bárbara"), "Carteirinha recupera seu leitor");
Carteirinha duplicada = new Carteirinha();
duplicada.setLeitor(leitor); duplicada.setNumero("C002"); duplicada.setEmitidaEm("2026-09-09");
try { db.dao(Carteirinha.class).create(duplicada); throw new AssertionError("Aceitou segunda carteirinha"); }
catch (SQLException esperado) { System.out.println("OK: segunda carteirinha rejeitada"); }

OK: Carteirinha recupera seu leitor


OK: segunda carteirinha rejeitada


## Relações 1:N e N:M
Dois livros e dois autores demonstram a autoria nos dois sentidos; um livro possui dois exemplares.

In [4]:
Livro livro = new Livro(); livro.setTitulo("Algoritmos em conjunto"); livro.setIsbn("9780000000001");
db.dao(Livro.class).create(livro);
Livro outroLivro = new Livro(); outroLivro.setTitulo("Dados em conjunto"); outroLivro.setIsbn("9780000000002");
db.dao(Livro.class).create(outroLivro);
Autor ana = new Autor(); ana.setNome("Ana Exemplo"); db.dao(Autor.class).create(ana);
Autor bruno = new Autor(); bruno.setNome("Bruno Exemplo"); db.dao(Autor.class).create(bruno);
for (Livro obra : java.util.List.of(livro, outroLivro)) {
    for (Autor autor : java.util.List.of(ana, bruno)) {
        LivroAutor autoria = new LivroAutor(); autoria.setLivro(obra); autoria.setAutor(autor);
        db.dao(LivroAutor.class).create(autoria);
    }
}
Exemplar exemplar = new Exemplar(); exemplar.setLivro(livro); exemplar.setCodigo("EX001"); db.dao(Exemplar.class).create(exemplar);
Exemplar copia = new Exemplar(); copia.setLivro(livro); copia.setCodigo("EX002"); db.dao(Exemplar.class).create(copia);
verificar(db.dao(Exemplar.class).queryForEq("livro_id", livro.getId()).size() == 2, "Livro possui dois exemplares");
verificar(db.dao(LivroAutor.class).queryForEq("livro_id", livro.getId()).size() == 2, "Livro possui dois autores");
verificar(db.dao(LivroAutor.class).queryForEq("autor_id", ana.getId()).size() == 2, "Autora participa de dois livros");

OK: Livro possui dois exemplares


OK: Livro possui dois autores


OK: Autora participa de dois livros


## Empréstimo, restrições e devolução
O mesmo exemplar só fica disponível para um novo empréstimo após a devolução.

In [5]:
Emprestimo emprestimo = servico.emprestar(leitor, exemplar, LocalDate.of(2026,9,9), LocalDate.of(2026,9,16));
try { servico.emprestar(leitor, exemplar, LocalDate.of(2026,9,9), LocalDate.of(2026,9,16)); throw new AssertionError("Aceitou empréstimo duplicado"); }
catch (SQLException esperado) { System.out.println("OK: exemplar já emprestado foi bloqueado"); }
servico.devolver(emprestimo.getId(), LocalDate.of(2026,9,12));
verificar(db.dao(Emprestimo.class).queryForId(emprestimo.getId()).getDevolvidoEm().equals("2026-09-12"), "Devolução persistida");
servico.emprestar(leitor, exemplar, LocalDate.of(2026,9,13), LocalDate.of(2026,9,20));
verificar(db.dao(Emprestimo.class).queryForEq("exemplar_id", exemplar.getId()).size() == 2, "Histórico preserva dois empréstimos");

OK: exemplar já emprestado foi bloqueado


OK: Devolução persistida


OK: Histórico preserva dois empréstimos


## Atualização, exclusão e integridade referencial

In [6]:
leitor.setNome("Bárbara Nogueira"); db.dao(Leitor.class).update(leitor);
verificar(db.dao(Leitor.class).queryForId(leitor.getId()).getNome().equals("Bárbara Nogueira"), "Nome atualizado");
Autor temporario = new Autor(); temporario.setNome("Cadastro temporário"); db.dao(Autor.class).create(temporario);
db.dao(Autor.class).delete(temporario);
verificar(db.dao(Autor.class).queryForId(temporario.getId()) == null, "Autor sem vínculos excluído");
try { db.dao(Leitor.class).delete(leitor); throw new AssertionError("Excluiu leitor com vínculos"); }
catch (SQLException esperado) { System.out.println("OK: exclusão de leitor vinculado bloqueada"); }

OK: Nome atualizado


OK: Autor sem vínculos excluído


OK: exclusão de leitor vinculado bloqueada


## Persistência após fechar e reabrir
Esta consulta usa outra conexão, comprovando que os dados estão no arquivo SQLite.

In [7]:
int leitorId = leitor.getId();
db.close();
db = new Database(arquivo);
verificar(db.dao(Leitor.class).queryForId(leitorId).getNome().equals("Bárbara Nogueira"), "Dados preservados após reabrir o banco");
verificar(db.dao(Emprestimo.class).countOf() == 2, "Histórico preservado em disco");
db.close();
System.out.println("Testes concluídos. Banco disponível em: " + arquivo);

OK: Dados preservados após reabrir o banco


OK: Histórico preservado em disco


Testes concluídos. Banco disponível em: /tmp/biblioteca-8891469874607388262.db
